Notebook para treinar o professor no Kaggle (ver README, seção Kaggle).

Antes de rodar: em **Settings**, escolha **Accelerator: GPU T4 x2** (ou P100) e ligue **Internet** (para baixar os pesos pré-treinados). Em **Add Input**, anexe os três datasets:
- `andrewmvd/isic-2019` (imagens de treino e validação)
- `lesoes-pele-meta` (rótulos, splits e código; privado)
- `lesoes-pele-extra` (imagens de teste e sem rótulo; privado)

# Ambiente

In [ ]:
import hashlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers, ops, applications, callbacks, losses, metrics, optimizers

from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

SEED = 42
keras.utils.set_random_seed(SEED)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if not gpus:
    print("ATENÇÃO: sem GPU. Em Settings > Accelerator, escolha GPU T4 x2 ou P100.")

# Dados

In [ ]:
# Os datasets anexados ficam em /kaggle/input, cada um com sua estrutura de pastas;
# as imagens são localizadas pelo nome do arquivo.
INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working"))

# pasta do dataset lesoes-pele-meta (followlinks: /kaggle/input pode ter links simbólicos)
META = next(Path(d) for d, _, files in os.walk(INPUT, followlinks=True) if "md5.csv" in files)
sys.path.insert(0, str(META))
from prepare_data import CLASSES, HAM_CLASSES
from pipeline import CLASS_IDX, HAM_IDX, class_weights, locate_images, make_datasets

df = locate_images(pd.read_csv(META / "labels.csv", low_memory=False), INPUT)
missing = df[df["image_path"].isna()]
print("imagens por split:", df["split"].value_counts().to_dict())
print("faltando:", missing["split"].value_counts().to_dict())
assert missing.empty, "Faltam imagens: confira se os três datasets estão anexados (ver topo)"

In [ ]:
# Confere por md5 que as imagens são as mesmas da base local (amostra por split)
N_CHECK = 100


def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


expected = pd.read_csv(META / "md5.csv").set_index("file_id")["md5"]
sample = df.groupby("split").sample(N_CHECK, random_state=SEED)
wrong = [f for f, p in zip(sample["file_id"], sample["image_path"]) if md5(p) != expected[f]]
print(f"{len(sample) - len(wrong)}/{len(sample)} imagens idênticas")
assert not wrong, f"Imagens diferentes da base local: {wrong[:5]}"

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

ds = make_datasets(df, img_size=IMG_SIZE, batch_size=BATCH_SIZE)
class_weight = class_weights(df.loc[df["split"] == "train", "dx"])

images, labels = next(iter(ds["train"]))
print("batch:", images.shape, labels.shape)
print("pesos:", {CLASSES[i]: round(w, 2) for i, w in class_weight.items()})

# Modelo professor

# Treino

# Teste

# Exportação

# Registro de iterações